In [1]:

import ultralytics
import roboflow
import cv2
import torch
import os
import yaml


In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("prince-suman-3ohwu").project("hairnet-detection-vktcj")
version = project.version(2)
dataset = version.download("yolov8")


In [38]:
from pathlib import Path
from collections import Counter

counter = Counter()
for split in ["train", "valid", "test"]:
    lbl_dir = Path("Hairnet-Detection-2") / split / "labels"
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().splitlines():
            if line.strip():
                counter[line.split()[0]] += 1

print(counter)

Counter({'3': 8123, '1': 8042})


In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("artyom").project("gloves-4nvvi")
version = project.version(1)
dataset = version.download("yolov8")


In [5]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("blazed").project("final-work-nyuq2-fuyr5")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Final-work-1 in yolov8:: 100%|██████████| 6903/6903 [00:07<00:00, 887.50it/s] 


In [6]:


from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
project = rf.workspace("blazed").project("gp-fqlbs-rcqzy")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to GP-1 in yolov8:: 100%|██████████| 6093/6093 [00:04<00:00, 1264.30it/s]


In [1]:
import os
import shutil
import yaml
from pathlib import Path
from ultralytics import YOLO

# ── Merge ────────────────────────────────────────────────────────────────────

DATASETS = ["Final-work-1", "GP-1", "gloves-1"]
MERGED   = Path("merged")
CLASSES  = ['glove', 'hairnet', 'no_glove', 'no_hairnet']

import shutil as sh
if MERGED.exists():
    sh.rmtree(MERGED)

for split in ["train", "valid", "test"]:
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    for split in ["train", "valid", "test"]:
        img_dir = Path(ds) / split / "images"
        lbl_dir = Path(ds) / split / "labels"
        if not img_dir.exists():
            continue
        for img in img_dir.glob("*"):
            dst_name = f"{ds}_{img.name}"
            shutil.copy(img, MERGED / split / "images" / dst_name)
            lbl = lbl_dir / (img.stem + ".txt")
            if lbl.exists():
                shutil.copy(lbl, MERGED / split / "labels" / (Path(dst_name).stem + ".txt"))

# Write unified data.yaml
data_yaml = {
    'path': str(MERGED.resolve()),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 4,
    'names': CLASSES
}
with open(MERGED / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# Count per split
for split in ["train", "valid", "test"]:
    n = len(list((MERGED / split / "images").glob("*")))
    print(f"{split}: {n} images")

print("\n✅ Merge complete")

# ── Train ────────────────────────────────────────────────────────────────────

model = YOLO("yolov8n.pt")

model.train(
    data=str((MERGED / "data.yaml").resolve()),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="runs/ppe",
    name="merged_v2",
    pretrained=True,
    plots=False,
    workers=0
)

train: 5005 images
valid: 1316 images
test: 673 images

✅ Merge complete
New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\merged\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000161D049A4A0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

In [3]:
import pandas as pd

df = pd.read_csv("runs/detect/runs/ppe/merged_v1/results.csv")
df.columns = df.columns.str.strip()
print(df[['epoch', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)']].tail(10))

    epoch  metrics/mAP50(B)  metrics/precision(B)  metrics/recall(B)
90     91           0.86348               0.86946            0.81827
91     92           0.86437               0.86618            0.81828
92     93           0.86288               0.87224            0.81640
93     94           0.86618               0.87514            0.82017
94     95           0.86679               0.87279            0.82435
95     96           0.86665               0.87182            0.82311
96     97           0.86644               0.86993            0.82629
97     98           0.86931               0.87087            0.82907
98     99           0.86913               0.86645            0.82961
99    100           0.86920               0.87066            0.82729


In [39]:
import os
import shutil
import yaml
from pathlib import Path
from ultralytics import YOLO
# dah el 4 merged 3ala yolo26
# ── Merge ────────────────────────────────────────────────────────────────────

DATASETS = ["Final-work-1", "GP-1", "gloves-1", "Hairnet-Detection-2"]  # ← added
MERGED   = Path("merged4sets")
CLASSES  = ['glove', 'hairnet', 'no_glove', 'no_hairnet']

import shutil as sh
if MERGED.exists():
    sh.rmtree(MERGED)

for split in ["train", "valid", "test"]:
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    for split in ["train", "valid", "test"]:
        img_dir = Path(ds) / split / "images"
        lbl_dir = Path(ds) / split / "labels"
        if not img_dir.exists():
            continue
        for img in img_dir.glob("*"):
            dst_name = f"{ds}_{img.name}"
            shutil.copy(img, MERGED / split / "images" / dst_name)
            lbl = lbl_dir / (img.stem + ".txt")
            if lbl.exists():
                shutil.copy(lbl, MERGED / split / "labels" / (Path(dst_name).stem + ".txt"))

# Write unified data.yaml
data_yaml = {
    'path': str(MERGED.resolve()),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 4,
    'names': CLASSES
}
with open(MERGED / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# Count per split
for split in ["train", "valid", "test"]:
    n = len(list((MERGED / split / "images").glob("*")))
    print(f"{split}: {n} images")

print("\nMerge complete")

# ── Train ────────────────────────────────────────────────────────────────────

model = YOLO("yolo26n.pt")

model.train(
    data=str((MERGED / "data.yaml").resolve()),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="runs/ppe",
    name="merged_v4",   # ← changed from merged_v2
    pretrained=True,
    plots=False,
    workers=4
)

train: 10969 images
valid: 1891 images
test: 959 images

Merge complete
New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\merged4sets\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_r

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BD89AAD550>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/runs/ppe/merged_v4/weights/best.pt")
metrics = model.val(data=r"C:\Users\ziada\PycharmProjects\CVproject\merged4sets\data.yaml")

Ultralytics 8.4.33  Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 67.214.3 MB/s, size: 48.7 KB)
val: Scanning C:\Users\ziada\PycharmProjects\CVproject\merged4sets\valid\labels.cache... 1891 images, 79 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1891/1891 360.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 7.2it/s 16.6s0.1s
                   all       1891       5505      0.854      0.823      0.865      0.455
                 glove        384        868      0.823      0.756      0.823      0.447
               hairnet        894       1818      0.916       0.95      0.962      0.578
              no_glove        575       1240      0.802      0.707      0.773      0.337
            no_hairnet        778       1579      0.875       0.88    

In [ ]:
import os
import yaml

# Fix hairnet yaml
hairnet_path = os.path.abspath("Hairnet-Detection-2")
hairnet_yaml = {
    'names': ['No-Hairnet', 'hairnet'],
    'nc': 2,
    'train': os.path.join(hairnet_path, 'train', 'images'),
    'val': os.path.join(hairnet_path, 'valid', 'images'),
    'test': os.path.join(hairnet_path, 'test', 'images')
}
with open("Hairnet-Detection-2/data.yaml", "w") as f:
    yaml.dump(hairnet_yaml, f)

# Fix gloves yaml
gloves_path = os.path.abspath("gloves-1")
gloves_yaml = {
    'names': ['glove'],
    'nc': 1,
    'train': os.path.join(gloves_path, 'train', 'images'),
    'val': os.path.join(gloves_path, 'valid', 'images'),
    'test': os.path.join(gloves_path, 'test', 'images')
}
with open("gloves-1/data.yaml", "w") as f:
    yaml.dump(gloves_yaml, f)

print("Hairnet dataset path:", hairnet_path)
print("Gloves dataset path:", gloves_path)
print("YAML files fixed!")

In [8]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
hairnet_model = YOLO('yolov8n.pt')

# Train on hairnet dataset
hairnet_results = hairnet_model.train(
    data=os.path.abspath("Hairnet-Detection-2/data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    name='hairnet_model',
    device=0,  # GPU
    patience=15,
    save=True,
    plots=True
)

print("Hairnet training complete!")
print("Best model saved at: runs/detect/hairnet_model/weights/best.pt")

Ultralytics 8.4.33  Python-3.14.0 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\Hairnet-Detection-2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hairnet_model, nbs=64, nms=False, opset=None, optimize

In [2]:
from ultralytics import YOLO

gloves_model = YOLO('yolov8n.pt')


gloves_results = gloves_model.train(
    data=os.path.abspath("gloves-1/data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,
    name='gloves_model',
    device=0,
    patience=15,
    save=True,
    plots=True
)

print("Gloves training complete!")
print("Best model saved at: runs/detect/gloves_model/weights/best.pt")

Ultralytics 8.4.33  Python-3.14.0 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ziada\PycharmProjects\CVproject\gloves-1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gloves_model, nbs=64, nms=False, opset=None, optimize=False, opti

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')  # prevents window popup, saves to file instead
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load both trained models
hairnet_model = YOLO('runs/detect/hairnet_model/weights/best.pt')
gloves_model = YOLO('runs/detect/gloves_model/weights/best.pt')

def check_cook(image_path):
    hairnet_results = hairnet_model(image_path, conf=0.3)
    gloves_results  = gloves_model(image_path, conf=0.3)

    # Check hairnet — class 1 = hairnet, class 0 = No-Hairnet
    wearing_hairnet = False
    for r in hairnet_results:
        for box in r.boxes:
            if int(box.cls) == 1:
                wearing_hairnet = True

    # Check gloves — class 0 = glove, need at least 2 detected
    glove_count = 0
    for r in gloves_results:
        for box in r.boxes:
            if int(box.cls) == 0:
                glove_count += 1
    wearing_gloves = glove_count >= 2

    # Build output
    output = {
        "hairnet": wearing_hairnet,
        "gloves": wearing_gloves,
        "approved": wearing_hairnet and wearing_gloves,
        "message": ""
    }

    if wearing_hairnet and wearing_gloves:
        output["message"] = " Cook is wearing both hairnet and gloves. APPROVED."
    elif wearing_hairnet and not wearing_gloves:
        output["message"] = " Cook is wearing hairnet but NO gloves (or only one)."
    elif not wearing_hairnet and wearing_gloves:
        output["message"] = " Cook is wearing gloves but NO hairnet."
    else:
        output["message"] = " Cook is wearing NEITHER hairnet nor gloves."

    print(output["message"])
    print("Hairnet:", wearing_hairnet)
    print("Gloves :", wearing_gloves)
    print("Approved:", output["approved"])

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(hairnet_results[0].plot(), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Hairnet Detection")
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(gloves_results[0].plot(), cv2.COLOR_BGR2RGB))
    axes[1].set_title("Gloves Detection")
    axes[1].axis('off')
    plt.suptitle(output["message"], fontsize=12, fontweight='bold',
                 color='green' if output["approved"] else 'red')
    plt.tight_layout()
    plt.savefig('detection_result.png')
    plt.close()
    print("Result saved to: detection_result.png")

    return output

root = tk.Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 1 No-Hairnet, 1 hairnet, 19.4ms
Speed: 3.7ms preprocess, 19.4ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 2 gloves, 20.6ms
Speed: 4.2ms preprocess, 20.6ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 480)
 Cook is wearing both hairnet and gloves. APPROVED.
Hairnet: True
Gloves : True
Approved: True
Result saved to: detection_result.png


In [16]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load the merged model
model = YOLO("runs/detect/runs/ppe/merged_v2/weights/best.pt")

# Class indices based on ['glove', 'hairnet', 'no_glove', 'no_hairnet']
CLASS_NAMES = ['glove', 'hairnet', 'no_glove', 'no_hairnet']

def check_cook(image_path):
    results = model(image_path, conf=0.5)
    r = results[0]

    detected = {name: 0 for name in CLASS_NAMES}
    for box in r.boxes:
        cls_name = CLASS_NAMES[int(box.cls)]
        detected[cls_name] += 1

    has_glove      = detected['glove'] > 0
    has_no_glove   = detected['no_glove'] > 0
    has_hairnet    = detected['hairnet'] > 0
    has_no_hairnet = detected['no_hairnet'] > 0

    # Decision logic — flag violations, require positive detection for approval
    gloves_ok  = has_glove and not has_no_glove
    hairnet_ok = has_hairnet and not has_no_hairnet

    approved = gloves_ok and hairnet_ok

    if not has_glove and not has_no_glove:
        gloves_status = "UNCERTAIN — hands not clearly visible"
    elif gloves_ok:
        gloves_status = "OK — gloves detected"
    else:
        gloves_status = "VIOLATION — bare hands detected"

    if not has_hairnet and not has_no_hairnet:
        hairnet_status = "UNCERTAIN — head not clearly visible"
    elif hairnet_ok:
        hairnet_status = "OK — hairnet detected"
    else:
        hairnet_status = "VIOLATION — no hairnet detected"

    output = {
        "gloves_ok": gloves_ok,
        "hairnet_ok": hairnet_ok,
        "approved": approved,
        "gloves_status": gloves_status,
        "hairnet_status": hairnet_status,
        "raw_detections": detected
    }

    print("Gloves :", output["gloves_status"])
    print("Hairnet:", output["hairnet_status"])
    print("Approved:", approved)
    print("Raw detections:", detected)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    title = "APPROVED" if approved else "REJECTED"
    color = 'green' if approved else 'red'
    plt.title(f"{title}\n{gloves_status}\n{hairnet_status}", fontsize=11, fontweight='bold', color=color)
    plt.tight_layout()
    plt.savefig('detection_result_mergedv2.png')
    plt.close()
    print("Result saved to: detection_result.png")

    return output

root = tk.Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 (no detections), 10.0ms
Speed: 2.3ms preprocess, 10.0ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Gloves : UNCERTAIN — hands not clearly visible
Hairnet: UNCERTAIN — head not clearly visible
Approved: False
Raw detections: {'glove': 0, 'hairnet': 0, 'no_glove': 0, 'no_hairnet': 0}
Result saved to: detection_result.png


In [15]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load the merged model
model = YOLO("runs/detect/runs/ppe/merged_v1/weights/best.pt")

# Class indices based on ['glove', 'hairnet', 'no_glove', 'no_hairnet']
CLASS_NAMES = ['glove', 'hairnet', 'no_glove', 'no_hairnet']

def check_cook(image_path):
    results = model(image_path, conf=0.5)
    r = results[0]

    detected = {name: 0 for name in CLASS_NAMES}
    for box in r.boxes:
        cls_name = CLASS_NAMES[int(box.cls)]
        detected[cls_name] += 1

    has_glove      = detected['glove'] > 0
    has_no_glove   = detected['no_glove'] > 0
    has_hairnet    = detected['hairnet'] > 0
    has_no_hairnet = detected['no_hairnet'] > 0

    # Decision logic — flag violations, require positive detection for approval
    gloves_ok  = has_glove and not has_no_glove
    hairnet_ok = has_hairnet and not has_no_hairnet

    approved = gloves_ok and hairnet_ok

    if not has_glove and not has_no_glove:
        gloves_status = "UNCERTAIN — hands not clearly visible"
    elif gloves_ok:
        gloves_status = "OK — gloves detected"
    else:
        gloves_status = "VIOLATION — bare hands detected"

    if not has_hairnet and not has_no_hairnet:
        hairnet_status = "UNCERTAIN — head not clearly visible"
    elif hairnet_ok:
        hairnet_status = "OK — hairnet detected"
    else:
        hairnet_status = "VIOLATION — no hairnet detected"

    output = {
        "gloves_ok": gloves_ok,
        "hairnet_ok": hairnet_ok,
        "approved": approved,
        "gloves_status": gloves_status,
        "hairnet_status": hairnet_status,
        "raw_detections": detected
    }

    print("Gloves :", output["gloves_status"])
    print("Hairnet:", output["hairnet_status"])
    print("Approved:", approved)
    print("Raw detections:", detected)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    title = "APPROVED" if approved else "REJECTED"
    color = 'green' if approved else 'red'
    plt.title(f"{title}\n{gloves_status}\n{hairnet_status}", fontsize=11, fontweight='bold', color=color)
    plt.tight_layout()
    plt.savefig('detection_result_v1.png')
    plt.close()
    print("Result saved to: detection_result.png")

    return output

root = tk.Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 1 hairnet, 11.1ms
Speed: 3.2ms preprocess, 11.1ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 480)
Gloves : UNCERTAIN — hands not clearly visible
Hairnet: OK — hairnet detected
Approved: False
Raw detections: {'glove': 0, 'hairnet': 1, 'no_glove': 0, 'no_hairnet': 0}
Result saved to: detection_result.png


In [14]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')  # prevents window popup, saves to file instead
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load both trained models
hairnet_model = YOLO('runs/detect/hairnet_model/weights/best.pt')
gloves_model = YOLO('runs/detect/gloves_model/weights/best.pt')

def check_cook(image_path):
    hairnet_results = hairnet_model(image_path, conf=0.5)
    gloves_results  = gloves_model(image_path, conf=0.5)

    # Check hairnet — class 1 = hairnet, class 0 = No-Hairnet
    wearing_hairnet = False
    for r in hairnet_results:
        for box in r.boxes:
            if int(box.cls) == 1:
                wearing_hairnet = True

    # Check gloves — class 0 = glove, need at least 2 detected
    glove_count = 0
    for r in gloves_results:
        for box in r.boxes:
            if int(box.cls) == 0:
                glove_count += 1
    wearing_gloves = glove_count >= 2

    # Build output
    output = {
        "hairnet": wearing_hairnet,
        "gloves": wearing_gloves,
        "approved": wearing_hairnet and wearing_gloves,
        "message": ""
    }

    if wearing_hairnet and wearing_gloves:
        output["message"] = "✅ Cook is wearing both hairnet and gloves. APPROVED."
    elif wearing_hairnet and not wearing_gloves:
        output["message"] = "⚠️ Cook is wearing hairnet but NO gloves (or only one)."
    elif not wearing_hairnet and wearing_gloves:
        output["message"] = "⚠️ Cook is wearing gloves but NO hairnet."
    else:
        output["message"] = "❌ Cook is wearing NEITHER hairnet nor gloves."

    print(output["message"])
    print("Hairnet:", wearing_hairnet)
    print("Gloves :", wearing_gloves)
    print("Approved:", output["approved"])

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(hairnet_results[0].plot(), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Hairnet Detection")
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(gloves_results[0].plot(), cv2.COLOR_BGR2RGB))
    axes[1].set_title("Gloves Detection")
    axes[1].axis('off')
    plt.suptitle(output["message"], fontsize=12, fontweight='bold',
                 color='green' if output["approved"] else 'red')
    plt.tight_layout()
    plt.savefig('detection_result.png')
    plt.close()
    print("Result saved to: detection_result.png")

    return output

root = tk.Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 1 No-Hairnet, 41.8ms
Speed: 2.9ms preprocess, 41.8ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-05-14 at 6.37.05 PM.jpeg: 640x480 2 gloves, 20.3ms
Speed: 2.3ms preprocess, 20.3ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)
⚠️ Cook is wearing gloves but NO hairnet.
Hairnet: False
Gloves : True
Approved: False
Result saved to: detection_result.png


In [17]:
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# Load merged_v4
model = YOLO("runs/detect/runs/ppe/merged_v4/weights/best.pt")
CLASS_NAMES = ['glove', 'hairnet', 'no_glove', 'no_hairnet']

def check_cook(image_path):
    results = model(image_path, conf=0.3)
    r = results[0]

    detected = {name: 0 for name in CLASS_NAMES}
    for box in r.boxes:
        cls_name = CLASS_NAMES[int(box.cls)]
        detected[cls_name] += 1

    has_glove      = detected['glove'] > 0
    has_no_glove   = detected['no_glove'] > 0
    has_hairnet    = detected['hairnet'] > 0
    has_no_hairnet = detected['no_hairnet'] > 0

    gloves_ok  = has_glove and not has_no_glove
    hairnet_ok = has_hairnet and not has_no_hairnet

    approved = gloves_ok and hairnet_ok

    if not has_glove and not has_no_glove:
        gloves_status = "UNCERTAIN — hands not clearly visible"
    elif gloves_ok:
        gloves_status = "OK — gloves detected"
    else:
        gloves_status = "VIOLATION — bare hands detected"

    if not has_hairnet and not has_no_hairnet:
        hairnet_status = "UNCERTAIN — head not clearly visible"
    elif hairnet_ok:
        hairnet_status = "OK — hairnet detected"
    else:
        hairnet_status = "VIOLATION — no hairnet detected"

    print("Gloves :", gloves_status)
    print("Hairnet:", hairnet_status)
    print("Approved:", approved)
    print("Raw detections:", detected)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    title = "APPROVED" if approved else "REJECTED"
    color = 'green' if approved else 'red'
    plt.title(f"{title}\n{gloves_status}\n{hairnet_status}", fontsize=11, fontweight='bold', color=color)
    plt.tight_layout()
    plt.savefig('detection_result_v4.png')
    plt.close()
    print("Result saved to: detection_result_v4.png")

# Pick image
root = tk.Tk()
root.withdraw()
file_path = filedialog.askopenfilename(
    title="Select a photo of the cook",
    filetypes=[("Image files", "*.jpg *.jpeg *.png")]
)

if file_path:
    print(f"Testing with: {file_path}")
    check_cook(file_path)

    # Also run low-confidence diagnostic
    print("\n--- Low-confidence detections (conf=0.05) ---")
    results_low = model(file_path, conf=0.05)
    for box in results_low[0].boxes:
        print(CLASS_NAMES[int(box.cls)], f"conf={float(box.conf):.3f}")
else:
    print("No file selected.")

Testing with: C:/Users/ziada/Desktop/sewar lel project/WhatsApp Image 2026-06-13 at 9.42.00 AM (1).jpeg

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-06-13 at 9.42.00 AM (1).jpeg: 640x480 1 glove, 1 hairnet, 2 no_gloves, 14.6ms
Speed: 2.4ms preprocess, 14.6ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 480)
Gloves : VIOLATION — bare hands detected
Hairnet: OK — hairnet detected
Approved: False
Raw detections: {'glove': 1, 'hairnet': 1, 'no_glove': 2, 'no_hairnet': 0}
Result saved to: detection_result_v4.png

--- Low-confidence detections (conf=0.05) ---

image 1/1 C:\Users\ziada\Desktop\sewar lel project\WhatsApp Image 2026-06-13 at 9.42.00 AM (1).jpeg: 640x480 1 glove, 1 hairnet, 2 no_gloves, 16.5ms
Speed: 2.0ms preprocess, 16.5ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)
hairnet conf=0.847
glove conf=0.518
no_glove conf=0.309
no_glove conf=0.073
